In [3]:
import pandas as pd
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
#from pathlib import Path

print("Librerías cargadas correctamente")
print(f"Fecha de validación: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

Librerías cargadas correctamente
Fecha de validación: 2026-01-30 17:32:54


In [4]:
# === CARGAR DATOS ===

with open('../data/extraccion_historica.json', 'r') as f:
    games = json.load(f)

print(f"Total juegos en la extracción histórica: {len(games)}")
muestra = games[:100]  # Analizar 100
print(f"Analizando una muestra de {len(muestra)} juegos")

Total juegos en la extracción histórica: 137419
Analizando una muestra de 100 juegos


In [5]:
# === ANÁLISIS DE CAMPOS NO NULOS DE LA MUESTRA (100 GAMES) ===

def analizar_no_nulos(games):
   
    # Analiza qué porcentaje de juegos tienen cada campo
        
    campos_simples = [
        'id', 'name', 'slug', 'released', 'tba',
        'rating', 'rating_top', 'ratings_count', 'metacritic',
        'playtime', 'added', 'suggestions_count',
        'reviews_count', 'reviews_text_count',
        'background_image', 'updated'
    ]
    
    campos_complejos = [
        'ratings', 'added_by_status', 'platforms', 
        'parent_platforms', 'genres', 'stores', 'tags', 'esrb_rating'
    ]
    
    resultados = []
    
    # Analizar campos simples
    for campo in campos_simples:
        total = len(games)
        con_valor = 0

        for game in games:
            valor = game.get(campo)
            if valor is not None and valor != '':
                con_valor +=1
        
        sin_valor = total - con_valor
        porcentaje = (con_valor / total) * 100
        
        resultados.append({
            'campo': campo,
            'tipo': 'simple',
            'con_valor': con_valor,
            'sin_valor': sin_valor,
            'porcentaje_completo': porcentaje
        })
    
    # Analizar campos complejos
    for campo in campos_complejos:
        total = len(games)
        con_valor = 0

        for game in games:
            valor = game.get(campo)
            if valor is not None:
                con_valor += 1

        sin_valor = total - con_valor
        porcentaje = (con_valor / total) * 100
        
        resultados.append({
            'campo': campo,
            'tipo': 'complejo',
            'con_valor': con_valor,
            'sin_valor': sin_valor,
            'porcentaje_completo': porcentaje,
        })
    
    return pd.DataFrame(resultados)

# Ejecutar análisis
df_no_nulos = analizar_no_nulos(muestra)


print("=== ANÁLISIS DE CAMPOS NO NULOS DE LA MUESTRA (100 JUEGOS) ===\n")
print(df_no_nulos.to_string(index=False))


=== ANÁLISIS DE CAMPOS NO NULOS DE LA MUESTRA (100 JUEGOS) ===

             campo     tipo  con_valor  sin_valor  porcentaje_completo
                id   simple        100          0                100.0
              name   simple        100          0                100.0
              slug   simple        100          0                100.0
          released   simple        100          0                100.0
               tba   simple        100          0                100.0
            rating   simple        100          0                100.0
        rating_top   simple        100          0                100.0
     ratings_count   simple        100          0                100.0
        metacritic   simple          1         99                  1.0
          playtime   simple        100          0                100.0
             added   simple        100          0                100.0
 suggestions_count   simple        100          0                100.0
     reviews_

In [6]:
# === CARGAR DATOS: ÚLTIMOS 100 JUEGOS ===

with open('../data/extraccion_historica.json', 'r') as f:
    games = json.load(f)

print(f"Total juegos en la extracción histórica: {len(games)}")
muestra_ultimos_cien = games[-101:-1]  # Analizar los 100 últimos
print(f"Analizando una muestra de los últimos {len(muestra_ultimos_cien)} juegos")

Total juegos en la extracción histórica: 137419
Analizando una muestra de los últimos 100 juegos


In [7]:
# === ANÁLISIS DE CAMPOS NO NULOS DE LA MUESTRA ÚLTIMOS CIEN JUEGOS ===


# Ejecutar análisis
df_no_nulos = analizar_no_nulos(muestra_ultimos_cien)


print("=== ANÁLISIS DE CAMPOS NO NULOS DE LA MUESTRA ÚLTIMOS CIEN ===\n")
print(df_no_nulos.to_string(index=False))


=== ANÁLISIS DE CAMPOS NO NULOS DE LA MUESTRA ÚLTIMOS CIEN ===

             campo     tipo  con_valor  sin_valor  porcentaje_completo
                id   simple        100          0                100.0
              name   simple        100          0                100.0
              slug   simple        100          0                100.0
          released   simple        100          0                100.0
               tba   simple        100          0                100.0
            rating   simple        100          0                100.0
        rating_top   simple        100          0                100.0
     ratings_count   simple        100          0                100.0
        metacritic   simple          0        100                  0.0
          playtime   simple        100          0                100.0
             added   simple        100          0                100.0
 suggestions_count   simple        100          0                100.0
     reviews_

In [18]:
# ========================================
# VALIDACIÓN DE TIPOS DE DATOS
# ========================================

def validar_tipos_datos(games):
    
    # Valida que los datos sean del tipo correcto y estén en rango
    
    errores = []
    
    for game in games:
        game_id = game.get('id', 'UNKNOWN')
        game_name = game.get('name', 'UNKNOWN')[:30]
        
        # Validar rating (debe ser float entre 0-5)
        rating = game.get('rating')
        if rating is not None:
            if not isinstance(rating, (int, float)):
                errores.append(f"Game {game_id} ({game_name}): rating no es numérico = {rating}")
            elif rating < 0 or rating > 5:
                errores.append(f"Game {game_id} ({game_name}): rating fuera de rango = {rating}")
        
        # Validar metacritic (debe ser int entre 0-100)
        metacritic = game.get('metacritic')
        if metacritic is not None:
            if not isinstance(metacritic, int):
                errores.append(f"Game {game_id} ({game_name}): metacritic no es int = {metacritic}")
            elif metacritic < 0 or metacritic > 100:
                errores.append(f"Game {game_id} ({game_name}): metacritic fuera de rango = {metacritic}")
        
        # Validar fecha (debe ser formato YYYY-MM-DD)
        released = game.get('released')
        if released:
            try:
                datetime.strptime(released, '%Y-%m-%d')
            except:
                errores.append(f"Game {game_id} ({game_name}): fecha inválida = {released}")
        
        # Validar ratings_count (debe ser >= 0)
        ratings_count = game.get('ratings_count')
        if ratings_count is not None and ratings_count < 0:
            errores.append(f"Game {game_id} ({game_name}): ratings_count negativo = {ratings_count}")
        
        # Validar playtime (debe ser >= 0)
        playtime = game.get('playtime')
        if playtime is not None and playtime < 0:
            errores.append(f"Game {game_id} ({game_name}): playtime negativo = {playtime}")
    
    return errores

errores_tipo = validar_tipos_datos(muestra)

print('=== VALIDACIÓN DE TIPOS DE DATOS ===')

if len(errores_tipo) == 0:
    print('Todos los datos tienen tipos correctos y están en rango válido')
else:
    print(f'Se encontraron {len(errores_tipo)} errores\n')
    # Mostrar primeros 10
    for e in errores_tipo[:10]:
        print(f' - {e}')

    # Si hay más de 10:
    if len(errores_tipo) > 10:
        print(f'\n...y {len(errores_tipo) - 10} errores más')


=== VALIDACIÓN DE TIPOS DE DATOS ===
Todos los datos tienen tipos correctos y están en rango válido


In [19]:
print('=== VALIDACIÓN DE TIPOS DE DATOS MUESTRA ÚLTIMOS CIEN JUEGOS ===')

errores_tipo_u = validar_tipos_datos(muestra_ultimos_cien)

if len(errores_tipo_u) == 0:
    print('Todos los datos tienen tipos correctos y están en rango válido')
else:
    print(f'Se encontraron {len(errores_tipo_u)} errores\n')
    # Mostrar primeros 10
    for e in errores_tipo_u[:10]:
        print(f' - {e}')

    # Si hay más de 10:
    if len(errores_tipo_u) > 10:
        print(f'\n...y {len(errores_tipo_u) - 10} errores más')

=== VALIDACIÓN DE TIPOS DE DATOS MUESTRA ÚLTIMOS CIEN JUEGOS ===
Todos los datos tienen tipos correctos y están en rango válido


In [10]:
# === CREAR DATAFRAME PARA ANÁLISIS - ESTADÍSTICAS DESCRIPTIVAS ===

import pandas as pd

# Extraer campos clave a DataFrame
df_games = pd.DataFrame([
    {
        'id': g.get('id'),
        'name': g.get('name'),
        'released': g.get('released'),
        'rating': g.get('rating'),
        'metacritic': g.get('metacritic'),
        'ratings_count': g.get('ratings_count'),
        'playtime': g.get('playtime'),
        'added': g.get('added'),
        'num_genres': len(g.get('genres', [])),
        'num_platforms': len(g.get('platforms', []))if isinstance(g.get('platforms'), list) else 0,
        'num_parent_platforms': len(g.get('parent_platforms', [])),
        'num_tags': len(g.get('tags', [])),
        'num_stores': len(g.get('stores', []))if isinstance(g.get('stores'), list) else 0,
        'tiene_esrb': g.get('esrb_rating') is not None,
        'tiene_metacritic': g.get('metacritic') is not None
    }
    for g in muestra
])

print('=== ANÁLISIS DE LA MUESTRA - ESTADÍSTICAS DESCRIPTIVAS ===')
print(df_games[['rating', 'metacritic', 'ratings_count', 'playtime', 'added']].describe())
print()

=== ANÁLISIS DE LA MUESTRA - ESTADÍSTICAS DESCRIPTIVAS ===
           rating  metacritic  ratings_count    playtime       added
count  100.000000         1.0     100.000000  100.000000  100.000000
mean     0.143500        76.0       1.230000    0.070000   15.520000
std      0.714334         NaN       7.075145    0.325825   91.986877
min      0.000000        76.0       0.000000    0.000000    0.000000
25%      0.000000        76.0       0.000000    0.000000    0.000000
50%      0.000000        76.0       0.000000    0.000000    0.000000
75%      0.000000        76.0       0.000000    0.000000    0.000000
max      4.330000        76.0      62.000000    2.000000  743.000000



In [22]:

# === ANÁLISIS DE GÉNEROS ===

import pandas as pd

# Extraer géneros en una lista plana
generos_lista = []
for game in muestra:
    for genre in game.get('genres', []):
        if genre.get('name'):
            generos_lista.append(genre['name'])

# Crear Series y contar
generos_series = pd.Series(generos_lista)
top_genres = generos_series.value_counts().head(10)

print("=== TOP 10 GÉNEROS EN LA MUESTRA ====")
for i, (genre, count) in enumerate(top_genres.items(), 1):
    porcentaje = (count / len(muestra)) * 100
    print(f"{i:2}. {genre:25} → {count:3} juegos ({porcentaje:5.1f}%)")
print()

=== TOP 10 GÉNEROS EN LA MUESTRA ====
 1. Action                    →  20 juegos ( 20.0%)
 2. Adventure                 →  15 juegos ( 15.0%)
 3. RPG                       →  12 juegos ( 12.0%)
 4. Platformer                →  11 juegos ( 11.0%)
 5. Simulation                →  10 juegos ( 10.0%)
 6. Strategy                  →   8 juegos (  8.0%)
 7. Puzzle                    →   8 juegos (  8.0%)
 8. Shooter                   →   7 juegos (  7.0%)
 9. Arcade                    →   5 juegos (  5.0%)
10. Indie                     →   5 juegos (  5.0%)



In [24]:

# === VALIDACIÓN DE IDs ÚNICOS ===


print('=== VALIDACIÓN DE IDs ÚNICOS ===')

ids = [game.get('id') for game in muestra]
ids_unicos = len(set(ids))
total_juegos = len(ids)

print(f'Total de juegos: {total_juegos}')
print(f'IDs únicos: {ids_unicos}')

if ids_unicos == total_juegos:
    print('Todos los IDs son únicos')
else:
    duplicados = total_juegos - ids_unicos
    print(f'Hay {duplicados} IDs duplicados')
    


=== VALIDACIÓN DE IDs ÚNICOS ===
Total de juegos: 100
IDs únicos: 100
Todos los IDs son únicos


In [26]:
# === VALIDAR DUPLICADOS POR NOMBRE ===

print('=== VALIDACIÓN DE DUPLICADOS POR NOMBRE ===')

nombres = [game.get('name') for game in muestra if game.get('name')]
nombres_unicos = set(nombres)

total_nombres = len(nombres)
total_unicos = len(nombres_unicos)

if total_nombres == total_unicos:
    print(' No hay nombres duplicados')
else:
    cant_duplicados = total_nombres - total_unicos
    print(f' Hay {cant_duplicados} nombres que se repiten')
    print(f'   Total de juegos: {total_nombres}')
    print(f'   Nombres únicos: {total_unicos}')

print()

=== VALIDACIÓN DE DUPLICADOS POR NOMBRE ===
 No hay nombres duplicados



In [25]:

# === VALIDACIÓN CAMPOS QUE NO PUEDEN FALTAR ===

print('=== VALIDACIÓN CAMPOS QUE NO PUEDEN FALTAR ===')

# Campos que NUNCA pueden estar vacíos/None
campos_obligatorios = ['id', 'name']

juegos_invalidos = []

for game in muestra:
    game_id = game.get('id', 'UNKNOWN')
    game_name = game.get('name', 'UNKNOWN')[:30]
    
    for campo in campos_obligatorios:
        valor = game.get(campo)
        
        # Verificar si es None, vacío o solo espacios
        if valor is None or (isinstance(valor, str) and not valor.strip()):
            juegos_invalidos.append({
                'game_id': game_id,
                'game_name': game_name,
                'campo_faltante': campo
            })

if len(juegos_invalidos) == 0:
    print('Todos los juegos tienen los campos requeridos')
else:
    print(f'{len(juegos_invalidos)} juegos con campos requeridos faltantes:\n')
    
    for i, error in enumerate(juegos_invalidos[:10], 1):
        print(f"{i:2}. Juego {error['game_id']} ({error['game_name']})")
        print(f"    - Falta campo: {error['campo_faltante']}\n")
    
    if len(juegos_invalidos) > 10:
        print(f" ...y {len(juegos_invalidos) - 10} errores más")

print()

=== VALIDACIÓN CAMPOS QUE NO PUEDEN FALTAR ===
Todos los juegos tienen los campos requeridos

